In [5]:
pip install gymnasium matplotlib

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install "gymnasium[classic-control]

     --------------------------------------- 10.6/10.6 MB 20.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from NEAT1 import Genotipo, Poblacion

class PenduloInvertidoGenotipo(Genotipo):
    
    def __init__(self, num_entradas=None, num_salidas=1):
        super().__init__()
        self.num_entradas = num_entradas
        self.num_salidas = num_salidas
        self.fitness = 0.0
    
    def inicializar(self, num_entradas, num_salidas=1):
        self.num_entradas = num_entradas
        self.num_salidas = num_salidas
        return self
    
    def decision(self, observacion):
        salida = self.fenotipo(observacion)
        return 0 if salida[0] < 0.5 else 1   #izquierda->0, derecha->1
        #la salida continua pasa a ser binaria para determinar la accion del coche
    
    def evaluar(self, entorno, max_pasos=500, render=False):
        observacion, _ = entorno.reset()  #se reinicia el entorno
        pasos = 0
        for _ in range(max_pasos):
            accion = self.decision(observacion)
            observacion, recompensa, fracaso, maximo, _ = entorno.step(accion) #se ejecuta la accion
            pasos += 1
            if fracaso or maximo:
                break
        self.fitness = pasos
        return pasos
    
    def calcular_fitness(self, n_evaluaciones=3):
        entorno = gym.make("CartPole-v1")
        fitness_total = 0
        for _ in range(n_evaluaciones):
            fitness_total += self.evaluar(entorno, max_pasos=500)
        entorno.close()
        self.fitness = fitness_total / n_evaluaciones  
        return self.fitness

def poblacion_inicial(poblacion, num_entradas=4, num_salidas=1):
    for i in poblacion.individuos:
        i.num_entradas = num_entradas
        i.num_salidas = num_salidas
    return poblacion

def entrenar(num_generaciones=50, tam=150):
    print("\nEntrenamiento mediante el metodo NEAT\n")
    poblacion = Poblacion(N=tam, genotipos=lambda: PenduloInvertidoGenotipo(num_entradas=4, num_salidas=1), num_entrada=4, num_salida=1)
    mejor_fitnesshis = []
    fitness_medhis = []
    verdadero_campeon = None
    for generacion in range(num_generaciones):
        fitness_todos = []
        for i in poblacion.individuos:
            fitness = i.calcular_fitness(n_evaluaciones=3)
            fitness_todos.append(fitness)
        mejor = max(fitness_todos)
        fitness_med = sum(fitness_todos) / len(fitness_todos)
        mejor_fitnesshis.append(mejor)
        fitness_medhis.append(fitness_med)
        mejor_ind_gen = max(poblacion.individuos, key=lambda x: x.fitness)
        if verdadero_campeon is None or mejor_ind_gen.fitness > verdadero_campeon.fitness:
            import copy
            verdadero_campeon = copy.deepcopy(mejor_ind_gen)
        poblacion.especiacion(delta_t=3.0)
        if generacion % 5 == 0 or generacion == num_generaciones - 1:    
            print(f"En la generacion {generacion}, el fitness medio es {fitness_med:.1f}, siendo el mejor {mejor:.1f}")
        if mejor >= 500:
            print(f"El problema se ha resuelto en la generacion {generacion}")
            break
        if generacion < num_generaciones - 1:  
            poblacion.reproduccion() 
    print(" ")
    print(f"El fitness del mejor individuo es {verdadero_campeon.fitness:.1f}")
    print(f"Tiene {len(verdadero_campeon.nodos)} nodos y {len(verdadero_campeon.conexiones)} conexiones")
    return verdadero_campeon, mejor_fitnesshis, fitness_medhis 


def visualizar_individuo(mejor_individuo, num_intentos=3, max_pasos=500):
    entorno = gym.make("CartPole-v1", render_mode="human")
    for inte in range(num_intentos):
        print(f"\n Intento {inte + 1}/{num_intentos}")
        observacion, _ = entorno.reset()
        pasos = 0
        recompensa_total = 0
        for _ in range(max_pasos):  
            accion = mejor_individuo.decision(observacion)    #se pone a prueba al mejor individuo nuevamente
            observacion, recompensa, fracaso, maximo, _ = entorno.step(accion)
            pasos += 1
            recompensa_total += recompensa
            if fracaso or maximo:
                break
        print(f"El numero de pasos dados sin que se caiga el pendulo es {pasos}")
    entorno.close()

def main():
    mejor_individuo, mejor_fithis, fitness_medhis = entrenar(num_generaciones=50,tam=150)
    visualizar_individuo(mejor_individuo, num_intentos=3)
    print(" ")
    print("Resumen")
    print(" ")
    print(f"Mejor fitness: {mejor_individuo.fitness:.1f}")
    nodos_entrada = sum(1 for n in mejor_individuo.nodos.values() if n.tipo == 'entrada')
    nodos_ocultos = sum(1 for n in mejor_individuo.nodos.values() if n.tipo == 'oculta')
    nodos_salida = sum(1 for n in mejor_individuo.nodos.values() if n.tipo == 'salida') 
    print(f"\nEstructura de la red neuronal:")
    print(f"  Nodos de entrada: {nodos_entrada}")
    print(f"  Nodos ocultos: {nodos_ocultos}")
    print(f"  Nodos de salida: {nodos_salida}")
    
if __name__ == "__main__":
    main()
    
#Entrenamiento mediante el metodo NEAT

#En la generacion 0, el fitness medio es 64.8, siendo el mejor 500.0
#El problema se ha resuelto en la generacion 0
 
#El fitness del mejor individuo es 500.0
#Tiene 5 nodos y 4 conexiones

# Intento 1/3
#El numero de pasos dados sin que se caiga el pendulo es 202

# Intento 2/3
#El numero de pasos dados sin que se caiga el pendulo es 500

# Intento 3/3
#El numero de pasos dados sin que se caiga el pendulo es 500
 
#Resumen
 
#Mejor fitness: 500.0

#Estructura de la red neuronal:
#  Nodos de entrada: 4
#  Nodos ocultos: 0
#  Nodos de salida: 1


Entrenamiento mediante el metodo NEAT

En la generacion 0, el fitness medio es 63.3, siendo el mejor 500.0
El problema se ha resuelto en la generacion 0
 
El fitness del mejor individuo es 500.0
Tiene 5 nodos y 4 conexiones

 Intento 1/3
El numero de pasos dados sin que se caiga el pendulo es 500

 Intento 2/3
El numero de pasos dados sin que se caiga el pendulo es 500

 Intento 3/3
El numero de pasos dados sin que se caiga el pendulo es 500
 
Resumen
 
Mejor fitness: 500.0

Estructura de la red neuronal:
  Nodos de entrada: 4
  Nodos ocultos: 0
  Nodos de salida: 1
